# BASIC RAG LANGCHAIN

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
unstructured_api_key = os.getenv("UNSTRUCTURED_API_KEY")


print(openai_api_key)
print(pinecone_api_key)
print(unstructured_api_key)

In [64]:
PINECONE_INDEX_NAME = "instruction-manual-rag-langchain"
PINECONE_NAMESPACE = "manuals"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-5-nano"

In [46]:
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

client = OpenAI()
pc = Pinecone(api_key=pinecone_api_key)

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

In [65]:
import time

existing = [idx["name"] for idx in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    time.sleep(20)

index = pc.Index(PINECONE_INDEX_NAME)

In [49]:
MANUAL_PATH = r"C:\Users\Muneeb Syed\Desktop\RAG\RAG FOR PRODUCTION\document.pdf"

In [50]:
from pathlib import Path

assert Path(MANUAL_PATH).exists()

In [39]:
from pathlib import Path
from unstructured_client import UnstructuredClient
from unstructured_client.models import shared, operations
from unstructured_client.models.shared import Files, PartitionParameters, Strategy

uc = UnstructuredClient(
    api_key_auth=unstructured_api_key,
)

with open(MANUAL_PATH, "rb") as f:
    partition_response = uc.general.partition(
        request=operations.PartitionRequest(
            partition_parameters=PartitionParameters(
                files=Files(content=f.read(), file_name=Path(MANUAL_PATH).name),
                strategy=Strategy.HI_RES,
                chunking_strategy="by_title",
                max_characters=1500,
                new_after_n_chars=1000,
                combine_under_n_chars=500,
                split_pdf_page=True,
                split_pdf_allow_failed=True,
                split_pdf_concurrency_level=15,
            )
        )
    )

chunks = partition_response.elements

INFO: split_pdf event=plan_created operation_id=e8b86fa1-40eb-474b-b8d9-adc58fc45ab6 filename=document.pdf strategy=hi_res page_range=1-13 page_count=13 split_size=2 chunk_count=7 concurrency=15 allow_failed=True cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=custom-ca-bundle mtls_cert=none
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=e8b86fa1-40eb-474b-b8d9-adc58fc45ab6 chunk_count=7 concurrency=15 allow_failed=True client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: P

37

In [68]:
from langchain_core.documents import Document

docs = []
id = 0
for i, ch in enumerate(chunks):
    text = ch.text if hasattr(ch, "text") else str(ch)
    metadata = {
        "source": MANUAL_PATH,
        "chunk_id": i,
    }
    docs.append(Document(page_content=text, metadata=metadata,id=f"doc_{id}"))
    id = id + 1

len(docs), docs[0].page_content[:500]

(37,
 "{'type': 'CompositeElement', 'element_id': '4ece5b4edd049d8300397afced3b751f', 'text': 'Computer Troubleshooting Part (1)\\n\\nBasic Troubleshooting Techniques\\n\\nIf you use a computer for work or entertainment, it is likely you’ve experienced error messages or unexpected crashes. These problems are common, and the more time you spend using a computer, the more likely it is you will use troubleshooting techniques.\\n\\nComputer troubleshooting is not reserved for IT professionals. With the right kno")

In [69]:
vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=PINECONE_INDEX_NAME,
    namespace=PINECONE_NAMESPACE,
)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [70]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [71]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise assistant for instruction manuals. Answer only from the provided context. If the context is insufficient, say so."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

In [72]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [73]:
def ask_manual(question: str):
    return rag_chain.invoke(question)

In [75]:
resp = ask_manual("my computer wont turn on what should i do ?")
resp.content

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'Try these steps in order:\n\n1) Check the power cord. Make sure it’s securely plugged into the back of the computer and into the wall outlet. (Power button will not start – Solution 1)\n\n2) If it’s plugged into an outlet, verify the outlet is working by plugging in another device (e.g., a lamp). (Solution 2)\n\n3) If you’re using a surge protector, ensure it’s turned on. You may need to reset it by turning it off and back on, and you can test it with a lamp. (Solution 3)\n\n4) If you’re using a laptop, the battery may be drained. Plug the AC adapter into the wall and try to start the laptop. If it still won’t start, wait a few minutes and try again. (Solution 4)\n\n5) If none of the above works, try restarting the computer. (Restart the computer guidance)'